# Precision and coverage across five CIFAR-100 subsets

Each row reports micro precision among admitted predictions, equivalent here to accepted accuracy. Coverage is the fraction of test images admitted. The two calibration tracks apply their own frozen policies to the same images within each draw. All five draws are retained, including failed, incomplete and missing runs.

Use the same `SDM_SUBSET_OUTPUT` and `SDM_SUBSET_BENCHMARK` settings as the experiment notebooks. Smoke output checks the pipeline. Benchmark results are exploratory and apply to the sampled classes. Overlapping draws share some test images, and prior inspection of CIFAR-100 test data limits confirmatory claims.

In [ ]:
import os
from pathlib import Path
import sys

REPO = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'experiments' / 'cifar100_subsets.py').is_file())
sys.path.insert(0, str(REPO / 'experiments'))
from cifar100_subsets import OUTPUT

SMOKE = os.environ.get('SDM_SUBSET_BENCHMARK') != '1'
OUTPUT_ROOT = Path(os.environ.get('SDM_SUBSET_OUTPUT', OUTPUT)).expanduser()

from cifar100_subsets import METHODS, TRACKS, summary_rows
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

print(f"Population: {'smoke (pipeline check)' if SMOKE else 'official test (exploratory)'}")
summary = pd.DataFrame(summary_rows(OUTPUT_ROOT, smoke=SMOKE))
display(summary[['draw', 'seed', 'status', 'classes']].drop_duplicates().reset_index(drop=True))
problems = summary[summary.status.ne('complete')]
if not problems.empty:
    display(problems.reindex(columns=['draw', 'status', 'reason']).reset_index(drop=True))


In [ ]:
completed = summary[summary.status.eq('complete')]
if completed.empty:
    print('No completed draws in this mode.')
else:
    display(completed[['draw', 'track', 'method', 'examples', 'admitted', 'correct_admitted',
                       'coverage', 'accepted_accuracy']].reset_index(drop=True))

    labels = dict(zip(METHODS, ['Raw softmax', 'Temperature softmax', 'Identity SDM']))
    styles = dict(zip(METHODS, [('o', '#245890'), ('s', '#b55b18'), ('D', '#277961')]))
    offsets = dict(zip(METHODS, [-.16, 0., .16]))
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, layout='constrained')
    for col, track in enumerate(TRACKS):
        axes[0, col].set_title('25 calibration images/class' if col == 0 else '100 calibration images/class')
        for row, (metric, title) in enumerate((('accepted_accuracy', 'Precision among admissions'),
                                              ('coverage', 'Test images admitted'))):
            axis = axes[row, col]
            for method in METHODS:
                data = completed[completed.track.eq(track) & completed.method.eq(method)]
                marker, color = styles[method]
                axis.scatter(data.draw + offsets[method], data[metric].to_numpy(dtype=float), marker=marker, color=color,
                             label=labels[method], alpha=.80)
            axis.set(ylabel=title, ylim=(-.03, 1.03), xticks=np.arange(1, 6))
            axis.yaxis.set_major_formatter(PercentFormatter(1))
            axis.grid(axis='y', alpha=.25)
            if row == 1:
                axis.set_xlabel('Draw')
    axes[0, 1].legend(loc='lower right', frameon=False)
    fig.suptitle('CIFAR-100 subset predictions | ' + ('smoke pipeline check' if SMOKE else 'official test'))
    plot_path = OUTPUT_ROOT / ('smoke' if SMOKE else 'benchmark') / 'precision_and_coverage.png'
    fig.savefig(plot_path, dpi=150)
    plt.show()

A missing precision point means the draw is missing, incomplete, or has zero admissions for that method. The status and counts tables identify which case applies.